In [ ]:
import pandas as pd

# 1. Разрешить показывать все столбцы
pd.set_option('display.max_columns', None)

# 2. Разрешить показывать все строки
pd.set_option('display.max_rows', None)

# 3. Убрать ограничение по ширине (чтобы таблица не переносилась и не обрезалась)
pd.set_option('display.width', None)

# 4. Убрать ограничение на длину текста в ячейках (на всякий случай)
pd.set_option('display.max_colwidth', None)

import pandas as pd

# Сопоставление: уровень -> имя файла
file_map = {
    'low':  'low.csv',
    'mid':  'mid.csv',
    'high': 'high.csv'
}

dfs = []

for level, filename in file_map.items():
    # skiprows=1 пропускает строку с title/start timestamp/...
    df = pd.read_csv(filename, skiprows=1)

    # Добавляем колонку с уровнем
    df['level'] = level

    dfs.append(df)

# Склеиваем по строкам
combined = pd.concat(dfs, ignore_index=True)

combined.head(20)

combined_active = combined[combined['PM.Attention.IsActive'] == 1].copy()

combined_active.head()

print(combined_active.shape)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)
import matplotlib.pyplot as plt

# 1. Выбираем признаки
features = [
    "PM.Attention.Raw",
    "PM.Engagement.Raw",
    "PM.Excitement.Raw",
    "PM.Stress.Raw",
    "PM.Relaxation.Raw",
    "PM.Interest.Raw",
    "PM.LongTermExcitement"
]

target = "level"

# 2. Подготавливаем данные
data = combined_active[features + [target]].copy()

# Преобразуем признаки в числовой формат
for col in features:
    data[col] = pd.to_numeric(data[col], errors="coerce")

data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=[target])
data = data[data[target].isin(["low", "mid", "high"])]

X = data[features]
y = data[target]

# 3. Проверяем распределение классов
print("Количество объектов:", len(data))
print("\nРаспределение классов:")
print(y.value_counts())
print("\nДоля классов:")
print(y.value_counts(normalize=True).round(3))

# 4. Делим данные на train и test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 5. Создаём модель
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])


# 6. Обучаем
model.fit(X_train, y_train)

# 7. Предсказываем классы на тестовой выборке
y_pred = model.predict(X_test)


# 8. Оцениваем качество
print("\n=== Random Forest ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(
    f"Balanced Accuracy: "
    f"{balanced_accuracy_score(y_test, y_pred):.3f}"
)
print(
    f"Macro F1: "
    f"{f1_score(y_test, y_pred, average='macro'):.3f}"
)

print("\nОтчёт по классам:")
print(classification_report(
    y_test,
    y_pred,
    labels=["low", "mid", "high"],
    zero_division=0
))

# 9. Матрица ошибок
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=["low", "mid", "high"],
    display_labels=["low", "mid", "high"],
    cmap="Blues"
)
plt.title("Матрица ошибок Random Forest")
plt.tight_layout()
plt.show()

# 10. Важность признаков
rf = model.named_steps["classifier"]

importance = pd.Series(
    rf.feature_importances_,
    index=features
).sort_values(ascending=False)

print("\nВажность признаков:")
print(importance)

importance.sort_values().plot(kind="barh")
plt.title("Важность признаков")
plt.xlabel("Feature importance")
plt.tight_layout()
plt.show()


import joblib

# Сохраняем весь pipeline (и imputer, и обученный RandomForest внутри)
joblib.dump(model, "rf_active_model.joblib")
print("Модель сохранена: rf_active_model.joblib")

model_loaded = joblib.load("rf_active_model.joblib")

sample = pd.DataFrame([{
    "PM.Attention.Raw":      0.72,
    "PM.Engagement.Raw":     0.65,
    "PM.Excitement.Raw":     0.40,
    "PM.Stress.Raw":         0.30,
    "PM.Relaxation.Raw":     0.55,
    "PM.Interest.Raw":       0.60,
    "PM.LongTermExcitement": 0.45,
}])[features]

model.predict(sample)